In [ ]:
# Project Name: Next-Out
# Description: Summarize data from many files into a single Dataframe or Excel file.
# Copyright (c) 2024 Justin Edenbaum, Never Gray
#
# This file is licensed under the MIT License.
# You may obtain a copy of the license at https://opensource.org/licenses/MIT

from pathlib import Path
import pandas as pd

from NO_file_tools import read_no_file
from NO_visio import valid_simtime

In [134]:
# Function to process multiple files and dataframe types
def summarize_segment_data(settings):
    """
    Process multiple NO files and extract data from multiple dataframe types.
    
    Args:
        settings: Dictionary containing settings including 'ses_output_str' list of files
        segments: List of segment numbers to extract
        df_names: List of dataframe types to process (e.g., ["SSA", "SA"])
    
    Returns:
        DataFrame with multi-index (File_Name, Segment) containing data from all dataframes
    """
    all_data = []
    segment_numbers_2_lookup = settings.get('segment_numbers_2_lookup', [])
    df_names = settings.get('segment_data_2_lookup', ["SSA"])
    # Process each .no file from settings
    for no_file_path_str in settings['ses_output_str']:
        try:
            no_file_path = Path(no_file_path_str) # Convert string path to Path object
            #TODO If list is output file, make it a NO file.
            data, _ = read_no_file(no_file_path) # Read the file
            file_df = None
            requested_time = settings.get('sim_time', -1)

            for df_name in df_names:
                if df_name not in data:
                    print(f"  Warning: {df_name} not found in {no_file_path.name}")
                    continue
                    
                try:
                    valid_time = valid_simtime(requested_time, data[df_name])
                    # Get the slice we want
                    df_slice = (data[df_name]
                              .xs(valid_time, level='Time')    # Remove time index
                              .loc[segment_numbers_2_lookup])                  # Get specific segments
                    
                    # For first dataframe, create the base DataFrame
                    if file_df is None:
                        file_df = df_slice
                    # For subsequent dataframes, add new columns
                    else:
                        file_df = pd.concat([file_df, df_slice], axis=1)
                        
                except Exception as e:
                    print(f"  Error processing {df_name}: {str(e)}")
            
            if file_df is not None:
                all_data.append(pd.concat([file_df], keys=[no_file_path.stem]))
                
        except Exception as e:
            print(f"Error processing file {no_file_path_str}: {str(e)}")
    
    # Combine all files' data
    if all_data:
        return pd.concat(all_data)
    return pd.DataFrame()

In [135]:
def create_excel_summary(summary_data, output_path):
    """
    Create an Excel file from the summary data with clean formatting.
    
    Args:
        summary_data: DataFrame with multi-index (File_Name, Segment)
        output_path: Path object pointing to where the Excel file should be saved
    
    Returns:
        Path object to the created Excel file
    """
    # Create a flat version of the summary data
    flat_summary = (summary_data
                    .reset_index(level=[0,1])           # Convert both index levels to columns
                    .rename(columns={                    # Rename columns to be more readable
                        'level_0': 'File_Name',
                        'level_1': 'Segment'
                    }))

    # Save to Excel without the index column
    flat_summary.to_excel(output_path, index=False)

    ''
    'dd'
    
    return output_path

In [136]:
# Setup parameters
no_directory_path = Path("C:\\Simulations\\Test")
segment_numbers_to_lookup = [2, 4]
time_point = 5600

# List of files to process
SSA_Only = [
        'C:/Simulations/Test\\PT09-S1GM-011-R01.no',
        'C:/Simulations/Test\\PT09-S1GM-012-F-R01.no',
        'C:/Simulations/Test\\PT09-S1GM-012-R01.no',
        'C:/Simulations/Test\\PT09-S1GM-012-WG-F-R01.no',
        'C:/Simulations/Test\\PT09-S1GM-012-WM-F-R01.no',
        'C:/Simulations/Test\\PT09-S1GM-013-F-R01.no',
        'C:/Simulations/Test\\PT09-S1GM-013-R01.no',
        'C:/Simulations/Test\\PT09-S1GM-014-F-R01.no',
        'C:/Simulations/Test\\PT09-S1GM-014-R01.no',
        'C:/Simulations/Test\\PT09-S1GM-014-WG-F-R01.no',
        'C:/Simulations/Test\\PT09-S1GM-014-WM-F-R01.no',
        'C:/Simulations/Test\\PT09-S1MG-011-R01.no',
        'C:/Simulations/Test\\PT09-S1MG-011-WG-R01.no',
        'C:/Simulations/Test\\PT09-S1MG-012-F-R01.no',
        'C:/Simulations/Test\\PT09-S1MG-012-R01.no',
        'C:/Simulations/Test\\PT09-S1MG-012-WG-F-R01.no',
        'C:/Simulations/Test\\PT09-S1MG-012-WG-R01.no',
        'C:/Simulations/Test\\PT09-S1MG-012-WM-F-R01.no',
        'C:/Simulations/Test\\PT09-S1MG-013-F-R01.no'
    ]

two_dataframes = [
    'C:/Simulations/PT10-B046\\PT10-P201.no',
    'C:/Simulations/PT10-B046\\PT10-P202-every-second.no',
    'C:/Simulations/PT10-B046\\PT10-P202.no',
    'C:/Simulations/PT10-B046\\PT10-P210.no',
    'C:/Simulations/PT10-B046\\PT10-P220.no',
    'C:/Simulations/PT10-B046\\PT10-P230.no'
]

# Settings dictionary
settings = {
    'ses_output_str': SSA_Only,
    'visio_template': 'C:/Simulations/InWin Simulations/SES-202 R3 Result Network for SES-277.vsdx',
    'results_folder_str': None,
    'simtime': -1,
    'conversion': '',
    'output': ['Excel', '', '', '', '', '', 'visio_2_pdf', 'visio_2_png', 'visio_2_svg'],
    'file_type': 'no_file',
    'path_exe': 'C:/Simulations/_EXE/SESV6_32.exe',
    'segment_numbers_2_lookup': segment_numbers_to_lookup,
    'segment_data_2_lookup': ["SSA"]
}

# Process all files from settings, getting both SSA and SA data
summary_data = summarize_segment_data(settings=settings)

if not summary_data.empty:
    # Create and save Excel summary
    excel_path = no_directory_path / "summary_results.xlsx"
    output_file = create_excel_summary(summary_data, excel_path)
    print(f"\nSaved summary to: {output_file}")


Saved summary to: C:\Simulations\Test\summary_results.xlsx
